# RUN_BATCH — Orchestrator batch analisa PV per tanggal

Menjalankan template notebook (mis. `20260209stringmap_v1.5.ipynb`) untuk
banyak tanggal sekaligus **tanpa mengubah file template**.

**Dua mode pengisian tanggal (boleh dicampur):**
- **Manual** — isi `DATE_TO_URL` sendiri.
- **Auto-enumerate** — cukup beri link **folder bulanan**; subfolder hari
  (`ddmmyyyy` mis. `30012026`, atau `ddmmyy` mis. `151225`) dibaca otomatis
  via Google Drive API lalu dipetakan ke URL.

**Cara pakai:** buka di Colab -> edit sel **Config** -> **Runtime > Run all**.
Saat auto-enumerate, akan ada **1x klik izin Google** di awal, sisanya jalan sendiri.
Hasil utama tersimpan ke `Cek PV String/outputs/`.

> Catatan: gdown butuh folder hari ter-share "anyone with link". Drive API bisa
> melisting folder privat yang Anda punya akses, tapi *download*-nya tetap perlu publik.

In [ ]:
# ============================ CONFIG (EDIT DI SINI) ============================
REPO_DIR     = '/content/drive/MyDrive/Cek PV String'
TEMPLATE_NB  = '/content/drive/MyDrive/Cek PV String/notebook/20260209stringmap_v1.5.ipynb'  # GANTI

SKIP_IF_DONE = True   # lewati tanggal yang m2_findings_YYYYMMDD.xlsx-nya sudah ada
QUIET_PLOTS  = True   # jangan render heatmap saat batch (file output tidak terpengaruh)

# --- Mode MANUAL (boleh kosong kalau pakai auto-enumerate) -------------------
DATE_TO_URL = {
    # '2026-02-09': 'https://drive.google.com/drive/folders/17JX8...',
}

# --- Mode AUTO-ENUMERATE: cukup beri link folder BULANAN (boleh >1) ----------
MONTH_FOLDER_URLS = [
    # 'https://drive.google.com/drive/folders/<id-folder-Januari-2026>',
    # 'https://drive.google.com/drive/folders/<id-folder-Februari-2026>',
]
DATE_FROM = None   # 'YYYY-MM-DD' atau None  (batas bawah range, opsional)
DATE_TO   = None   # 'YYYY-MM-DD' atau None  (batas atas range, opsional)
# =============================================================================

In [ ]:
# =================== AUTO-ENUMERATE (jalan jika MONTH_FOLDER_URLS terisi) =====
import re as _re
from datetime import datetime as _dt, date as _date

def _folder_id(url):
    m = _re.search(r'/folders/([A-Za-z0-9_-]+)', url) or _re.search(r'[?&]id=([A-Za-z0-9_-]+)', url)
    if not m:
        raise ValueError(f'Tidak bisa ambil folder id dari URL: {url}')
    return m.group(1)

def _parse_day(name):
    s = name.strip()
    if not s.isdigit():
        return None
    fmt = {8: '%d%m%Y', 6: '%d%m%y'}.get(len(s))   # 8=ddmmyyyy, 6=ddmmyy
    if not fmt:
        return None
    try:
        return _dt.strptime(s, fmt).date()
    except ValueError:
        return None

if MONTH_FOLDER_URLS:
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    _svc = build('drive', 'v3')

    _lo = _dt.strptime(DATE_FROM, '%Y-%m-%d').date() if DATE_FROM else _date.min
    _hi = _dt.strptime(DATE_TO,   '%Y-%m-%d').date() if DATE_TO   else _date.max

    _found, _skipped = {}, []
    for _murl in MONTH_FOLDER_URLS:
        _fid = _folder_id(_murl)
        _tok = None
        while True:
            _resp = _svc.files().list(
                q="'" + _fid + "' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false",
                fields='nextPageToken, files(id,name)', pageSize=1000, pageToken=_tok,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute()
            for _f in _resp.get('files', []):
                _d = _parse_day(_f['name'])
                if _d is None:
                    _skipped.append(_f['name']); continue
                if _lo <= _d <= _hi:
                    _found[_d.isoformat()] = 'https://drive.google.com/drive/folders/' + _f['id']
            _tok = _resp.get('nextPageToken')
            if not _tok:
                break

    # manual menang kalau bentrok; hasil diurutkan tanggal
    DATE_TO_URL = dict(sorted({**_found, **DATE_TO_URL}.items()))
    print(f'[enumerate] folder hari ketemu={len(_found)}  dilewati(nama bukan tanggal)={len(_skipped)}')
    if _skipped:
        print('  contoh dilewati:', _skipped[:5])
    print(f'[enumerate] total tanggal siap jalan = {len(DATE_TO_URL)}')
    for _k in list(DATE_TO_URL)[:3]:
        print('   ', _k, '->', DATE_TO_URL[_k])
else:
    print('[enumerate] MONTH_FOLDER_URLS kosong -> pakai DATE_TO_URL manual ('
          + str(len(DATE_TO_URL)) + ' tanggal)')

In [ ]:
# ============================ ENGINE (biarkan apa adanya) =====================
from google.colab import drive
drive.mount('/content/drive')

import json, os, sys, gc

if QUIET_PLOTS:
    import matplotlib
    matplotlib.use('Agg')        # batch: tidak render ke layar (file output tidak berubah)
import matplotlib.pyplot as plt

assert os.path.isdir(REPO_DIR),    f'REPO_DIR tidak ditemukan: {REPO_DIR}'
assert os.path.isfile(TEMPLATE_NB), f'TEMPLATE_NB tidak ditemukan: {TEMPLATE_NB}'
assert DATE_TO_URL, 'DATE_TO_URL kosong: isi manual atau set MONTH_FOLDER_URLS lalu run sel enumerate.'

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Nonaktifkan unduhan browser pada Cell 8 template (TANPA mengubah file template)
import google.colab.files as _gf
_gf.download = lambda *a, **k: print('   [skip files.download]', *a)

# Muat template sekali; ambil hanya code cell secara berurutan
_nb = json.load(open(TEMPLATE_NB, encoding='utf-8'))
CODE_CELLS = [''.join(c['source']) for c in _nb['cells'] if c['cell_type'] == 'code']
OUT_DIR = os.path.join(REPO_DIR, 'outputs')

def _already_done(datestr):
    ymd = datestr.replace('-', '')
    return os.path.isfile(os.path.join(OUT_DIR, f'm2_findings_{ymd}.xlsx'))

def _inject_url(src, url):
    out = []
    for line in src.split('\n'):
        if line.lstrip().startswith('DRIVE_FOLDER_URL') and '=' in line:
            indent = line[:len(line) - len(line.lstrip())]
            out.append(f'{indent}DRIVE_FOLDER_URL = "{url}"')
        else:
            out.append(line)
    return '\n'.join(out)

def run_one_day(url):
    g = {'__name__': '__main__'}          # namespace baru tiap hari (anti bocor state)
    for src in CODE_CELLS:
        exec(_inject_url(src, url), g)

ok = skip = fail = 0
for datestr, url in DATE_TO_URL.items():
    if SKIP_IF_DONE and _already_done(datestr):
        print(f'SKIP {datestr}: output sudah ada'); skip += 1; continue
    print(f'==================  {datestr}  ==================')
    try:
        run_one_day(url); print(f'OK   {datestr}'); ok += 1
    except Exception as e:
        print(f'FAIL {datestr}: {type(e).__name__}: {e}'); fail += 1
    finally:
        plt.close('all'); gc.collect()

print(f'\n=== RINGKASAN ===  sukses={ok}  skip={skip}  gagal={fail}  total={len(DATE_TO_URL)}')
print(f'Hasil: {OUT_DIR}/m2_findings_YYYYMMDD.xlsx (+ .jsonl, pr_daily_*.csv, baseline/)')